# Hung Experiment 2 — LGBM From Scratch
GBDT với histogram splitting + leaf-wise growth + softmax loss

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

In [ ]:
from src.core.trainer import Trainer
from src.model.lgbm_scratch import LGBMScratchModel

In [ ]:
train = pd.read_csv('../data/processed/tree/train.csv')
test  = pd.read_csv('../data/processed/tree/test.csv')
display(train.head())
display(test.head())

## Train thử 1 model — xem loss từng iter
Dùng n_estimators=20 để nhanh, thấy loss in ra rõ ràng.

In [ ]:
# Train thẳng 1 model — KHÔNG qua Trainer/CV — để xem loss từng iter
from src.core.trainer import Trainer
import numpy as np

drop_cols    = ['risk_class', 'risk_class_encoded', 'latitude', 'longitude']
feature_cols = [c for c in train.columns if c not in drop_cols]
X_train = train[feature_cols].values
y_train = train['risk_class_encoded'].values

model = LGBMScratchModel(
    n_estimators      = 20,   # nhỏ để nhanh, tăng sau
    learning_rate     = 0.1,
    max_depth         = 4,
    num_leaves        = 15,
    min_child_samples = 20,
    reg_lambda        = 1.0,
)
model.build()
model.fit(X_train, y_train)

## Vẽ loss curve

In [ ]:
loss_history = model.model.loss_history

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history,
         color='steelblue', linewidth=2, marker='o', markersize=4)
plt.xlabel('Iteration')
plt.ylabel('Cross-entropy Loss')
plt.title('GBDT Training Loss per Iteration')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Loss iter 1  : {loss_history[0]:.6f}')
print(f'Loss iter cuối: {loss_history[-1]:.6f}')
print(f'Giảm tổng    : {loss_history[0] - loss_history[-1]:.6f}')

## Tune hyperparameters — sau khi xác nhận model chạy đúng

In [ ]:
# Bỏ comment khi muốn tune — sẽ mất nhiều thời gian hơn
# trainer = Trainer(model=LGBMScratchModel(), n_iter=5, cv=5)
# trainer.tune(train)
# print('Best params:', trainer.model.best_params)

## Evaluate test — CHỈ CHẠY 1 LẦN

In [ ]:
from sklearn.metrics import fbeta_score, f1_score

X_test = test[feature_cols].values
y_test = test['risk_class_encoded'].values

y_pred = model.predict(X_test)

f2  = fbeta_score(y_test, y_pred, beta=2, average='macro')
f1w = f1_score(y_test, y_pred, average='weighted')
print(f'Test F2-macro   : {f2:.4f}')
print(f'Test F1-weighted: {f1w:.4f}')

## Save model

In [ ]:
import pickle
with open('../config/LGBMScratch.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Saved to ../config/LGBMScratch.pkl')